In [19]:
# imports + load the 4 CSVs, fix converted column, check shapes
import pandas as pd, numpy as np, statsmodels.api as sm
from scipy import stats
import seaborn as sns, matplotlib.pyplot as plt

experiments = pd.read_csv("../data/experiments.csv")
assignments = pd.read_csv("../data/assignments.csv")
events = pd.read_csv("../data/events.csv")
users = pd.read_csv("../data/users.csv")


print(experiments.shape, assignments.shape, events.shape, users.shape)

(10, 9) (93050, 5) (619048, 7) (46000, 8)


In [35]:
# keep only the 8 trustworthy experiments, then merge everything into one table
ANALYSABLE = ["EXP-001","EXP-002","EXP-003","EXP-004","EXP-006","EXP-007","EXP-008","EXP-009"]

assignments = assignments[assignments.experiment_id.isin(ANALYSABLE)]
events = events[events.experiment_id.isin(ANALYSABLE)]

merged = assignments.merge(users, on="user_id", how="left").merge(events, on=["experiment_id","user_id"], how="left")

print(merged.shape)
merged.head()

(476457, 17)


,assignment_id,experiment_id,user_id,variant,assigned_at,signup_date,country,device,acquisition_channel,pre_sessions_28d,pre_orders_28d,pre_revenue_28d,event_date,sessions,converted,orders,revenue_xaf
0,AS-0016202,EXP-002,UU736547,control,2025-11-08,2025-05-14,CF,ios,referral,4,4,3331,2025-11-04,3.0,False,0.0,0.0
1,AS-0016202,EXP-002,UU736547,control,2025-11-08,2025-05-14,CF,ios,referral,4,4,3331,2025-11-05,3.0,False,0.0,0.0
2,AS-0016202,EXP-002,UU736547,control,2025-11-08,2025-05-14,CF,ios,referral,4,4,3331,2025-11-06,1.0,False,0.0,0.0
3,AS-0016202,EXP-002,UU736547,control,2025-11-08,2025-05-14,CF,ios,referral,4,4,3331,2025-11-07,2.0,False,0.0,0.0
4,AS-0016202,EXP-002,UU736547,control,2025-11-08,2025-05-14,CF,ios,referral,4,4,3331,2025-11-10,2.0,False,0.0,0.0


In [36]:
# check for merge problems: missing device (bad merge to users) or missing converted (no matching event)
print(merged["device"].isna().sum())
print(merged["converted"].isna().sum())


0
117


In [37]:
print(len(merged))
print(476457 / len(merged))

476457
1.0


In [38]:
# compare actual values between the two tables to spot mismatches
print(assignments["user_id"].head(3).tolist())
print(events["user_id"].head(3).tolist())
print(assignments["experiment_id"].unique()[:3])
print(events["experiment_id"].unique()[:3])

['UU736547', 'UU730730', 'UU729627']
['UU714798', 'UU714798', 'UU714798']
['EXP-002' 'EXP-004' 'EXP-007']
['EXP-001' 'EXP-002' 'EXP-003']


In [24]:
# pick one real user from assignments and check if events has ANY row for that exact user+experiment
sample = assignments.iloc[0]
print(sample["user_id"], sample["experiment_id"])

match = events[(events.user_id == sample["user_id"]) & (events.experiment_id == sample["experiment_id"])]
print(len(match))


UU736547 EXP-002
10


In [25]:
# check for hidden whitespace or dtype mismatches between the join columns
print(repr(assignments["user_id"].iloc[0]))
print(repr(events["user_id"].iloc[0]))
print(assignments["user_id"].dtype, events["user_id"].dtype)
print(assignments["experiment_id"].dtype, events["experiment_id"].dtype)

'UU736547'
'UU714798'
object object
object object


In [26]:
# minimal test: merge assignments to events using ONLY that one known-good row
test_merge = assignments.iloc[[0]].merge(events, on=["experiment_id","user_id"], how="left")
print(len(test_merge))
print(test_merge[["user_id","experiment_id","event_date"]].head())

10
    user_id experiment_id  event_date
0  UU736547       EXP-002  2025-11-04
1  UU736547       EXP-002  2025-11-05
2  UU736547       EXP-002  2025-11-06
3  UU736547       EXP-002  2025-11-07
4  UU736547       EXP-002  2025-11-10


In [39]:
# check what's actually in your variables right now
print(len(assignments), len(events))
print(assignments.columns.tolist())
print(events.columns.tolist())

73618 475582
['assignment_id', 'experiment_id', 'user_id', 'variant', 'assigned_at']
['experiment_id', 'user_id', 'event_date', 'sessions', 'converted', 'orders', 'revenue_xaf']


In [40]:
# rerun the real merge fresh, and check 
merged = assignments.merge(users, on="user_id", how="left").merge(events, on=["experiment_id","user_id"], how="left")
print(merged["converted"].isna().sum(), len(merged))

117 476457


In [29]:
# do the merge in two separate visible steps instead of chaining
step1 = assignments.merge(users, on="user_id", how="left")
print(len(step1), step1["device"].isna().sum())

step2 = step1.merge(events, on=["experiment_id","user_id"], how="left")
print(len(step2), step2["converted"].isna().sum())


73618 0
476457 117


In [30]:
# check for any column name overlap between users and events besides the merge keys
print(set(users.columns) & set(events.columns))

{'user_id'}


In [31]:
# confirm step1's key columns still match assignments exactly
print(step1["user_id"].equals(assignments["user_id"].reset_index(drop=True)))
print(step1["experiment_id"].equals(assignments["experiment_id"].reset_index(drop=True)))
print(step1[["user_id","experiment_id"]].dtypes)
print(events[["user_id","experiment_id"]].dtypes)
sample = step1.iloc[0]
match = events[(events.user_id == sample["user_id"]) & (events.experiment_id == sample["experiment_id"])]
print(sample["user_id"], sample["experiment_id"], len(match))

True
True
user_id          object
experiment_id    object
dtype: object
user_id          object
experiment_id    object
dtype: object
UU736547 EXP-002 10


In [32]:
# reset indexes explicitly, then merge fresh and check right away
a = assignments.reset_index(drop=True)
e = events.reset_index(drop=True)
test = a.merge(e, on=["experiment_id","user_id"], how="inner")
print(len(test))


476340


In [33]:
# clean, confirmed-working merge
merged = assignments.merge(users, on="user_id", how="left").merge(events, on=["experiment_id","user_id"], how="left")
print(len(merged), merged["converted"].isna().sum())
print(events["converted"].head(10))
print(events["converted"].unique())

476457 117
0    False
1    False
2    False
3    False
4    False
5    False
6    False
7    False
8    False
9    False
Name: converted, dtype: bool
[False  True]


In [34]:
# check what converted actually looked like right after loading, before the broken .map() line
events2 = pd.read_csv("../data/events.csv")
print(events2["converted"].dtype)
print(events2["converted"].unique())

bool
[False  True]


In [41]:
# reload all 4 files clean (converted stays boolean automatically), then rebuild the merge
experiments = pd.read_csv("../data/experiments.csv")
assignments = pd.read_csv("../data/assignments.csv")
events = pd.read_csv("../data/events.csv")
users = pd.read_csv("../data/users.csv")

ANALYSABLE = ["EXP-001","EXP-002","EXP-003","EXP-004","EXP-006","EXP-007","EXP-008","EXP-009"]
assignments = assignments[assignments.experiment_id.isin(ANALYSABLE)]
events = events[events.experiment_id.isin(ANALYSABLE)]

merged = assignments.merge(users, on="user_id", how="left").merge(events, on=["experiment_id","user_id"], how="left")
print(len(merged), merged["converted"].isna().sum(), merged["device"].isna().sum())

476457 117 0


In [42]:
# isolate one experiment to build and test the regression code on
one_test = merged[merged.experiment_id == "EXP-001"].copy()
print(one_test.shape)
print(one_test["variant"].value_counts())

(57071, 17)
variant
control      28725
treatment    28346
Name: count, dtype: int64
